# 14. File I/O & Serialization (5+ Years Interview Guide)
Exhaustive revision guide to binary persistence (.npy), compressed multi-array archives (.npz), memmap, and text loading on raw_transactions.csv.

### Key 5-Year Interview Concepts Covered:
- **Binary Serialization (`.npy`)**: Dedicated cell for `np.save()` and `np.load()`.
- **Multi-Array Archives (`.npz`)**: Dedicated cell for `np.savez()` and `np.savez_compressed()`.
- **Memory-Mapped Files (`np.memmap`)**: Out-of-core virtual memory mapping for arrays larger than RAM.
- **Text Parsing**: Dedicated cell for `np.loadtxt()` and `np.genfromtxt()`.

This interactive revision guide loads and operates directly on `data/raw_transactions.csv` using dedicated cells per method.

In [ ]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

### Binary Persistence: `np.save()` & `np.load()`
**Explanation**: Serializes transaction amounts to fast `.npy` binary format.

**Syntax**: `np.save('scratch/tx_amounts.npy', amounts)` / `np.load('scratch/tx_amounts.npy')`

In [ ]:
os.makedirs('scratch', exist_ok=True)
np.save('scratch/tx_amounts.npy', amounts)
loaded_amounts = np.load('scratch/tx_amounts.npy')
print('Saved & Loaded array equal?:', np.array_equal(amounts, loaded_amounts))

### Uncompressed Archive: `np.savez()`
**Explanation**: Bundles amounts, fraud flags, and account ages into a single `.npz` archive.

**Syntax**: `np.savez('scratch/transactions_all.npz', amounts=amounts, fraud=fraud_flags)`

In [ ]:
np.savez('scratch/transactions_all.npz', amounts=amounts, fraud=fraud_flags, ages=account_ages)
with np.load('scratch/transactions_all.npz') as arch:
    print('Archived Keys:', arch.files)
    print('Loaded fraud array shape:', arch['fraud'].shape)

### Compressed Multi-Array Archive: `np.savez_compressed()`
**Explanation**: Applies zip compression saving 70% disk space.

**Syntax**: `np.savez_compressed('scratch/tx_compressed.npz', amounts=amounts, fraud=fraud_flags)`

In [ ]:
np.savez_compressed('scratch/tx_compressed.npz', amounts=amounts, fraud=fraud_flags)
print('Compressed npz written. Size:', os.path.getsize('scratch/tx_compressed.npz'), 'bytes')

### Out-of-Core Memory Mapping: `np.memmap()`
**Explanation**: Memory-maps binary transaction records directly into virtual RAM without loading file into heap.

**Syntax**: `np.memmap('scratch/large_tx.dat', dtype='float64', mode='w+', shape=amounts.shape)`

In [ ]:
mmap_tx = np.memmap('scratch/mmap_transactions.dat', dtype='float64', mode='w+', shape=amounts.shape)
mmap_tx[:] = amounts[:]
mmap_tx.flush()
print('Memory-mapped transaction buffer created. Shape:', mmap_tx.shape)

### Text Parsing: `np.loadtxt()` & `np.genfromtxt()`
**Explanation**: Parses comma-delimited numeric columns directly from CSV.

**Syntax**: `np.genfromtxt('data/raw_transactions.csv', delimiter=',', skip_header=1, usecols=(3, 7))`

In [ ]:
tx_numeric_cols = np.genfromtxt(csv_path, delimiter=',', skip_header=1, usecols=(3, 7), max_rows=5)
print('Parsed Numeric Columns (amount, account_age):\n', tx_numeric_cols)

## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: Streaming Chunked Aggregation on Memmap
**Explanation**: Calculate total transaction spend across memory-mapped file in chunks.

**Syntax**: `mmap_tx[start:end].sum()`

In [ ]:
chunk_sz = 2500
total_mmap_sum = sum(mmap_tx[i:i+chunk_sz].sum() for i in range(0, len(mmap_tx), chunk_sz))
print(f'Total Spend across Memmap Chunks: ${total_mmap_sum:,.2f}')